# 01 — Obnaruzhenie problem kachestva dannykh

**Datasest**: `data/processed/financial_news_merged.csv`  
**Zadacha**: binarnaya klassifikatsiya — `positive_market_impact` (0/1)

In [1]:
import sys, os
sys.path.insert(0, '..')
import warnings
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scripts.quality_utils import run_full_detection, save_report, print_report_summary

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('../data/reports', exist_ok=True)
print('OK')


OK


## 1. Zagruzka dannykh

In [2]:
df = pd.read_csv('../data/processed/financial_news_merged.csv', encoding='utf-8-sig')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)


Shape: (15040, 9)
Columns: ['title', 'body', 'date', 'source', 'sentiment_score', 'article_type', 'sectors', 'tickers', 'positive_market_impact']


,title,body,date,source,sentiment_score,article_type,sectors,tickers,positive_market_impact
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,rdv,-0.5,finance,['Energy'],['GAZP'],0.0
1,no title,""" Рейтинг акций комьюнити РДВ. #опрос """,2024-09-09,rdv,0.0,advertising,[],[],0.0
2,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",2024-09-09,rdv,-0.5,finance,"['Energy', 'Financial Service']",[],0.0


## 2. Polnaya proverka kachestva

In [3]:
report = run_full_detection(df, target_col='positive_market_impact')
print_report_summary(report)


Quality Score: 43/100
Dataset: 15040 rows × 9 cols

[ missing ]
  🟡 sentiment_score: 40 (0.27%)
  🟡 article_type: 40 (0.27%)
  🟡 sectors: 40 (0.27%)
  🟡 tickers: 40 (0.27%)
  🟡 positive_market_impact: 40 (0.27%)
[ duplicates ]
  🟠 exact_duplicates: 2 (0.01%)
  🟡 no_title_entries: 11021 (73.28%) — Telegram-style posts from rdv/t_invest/t_analytic — no title by design
[ text ]
  🟢 emoji_in_title: 5 (0.03%)
  🟡 short_body: 29 (0.19%)
  ✅ empty_body: 0
[ labels ]
  🟡 missing_labels: 40 (0.27%)
  ✅ class_balance: 
[ content ]
  🟡 advertising_articles: 1213 (8.07%) — May add noise to classification task
  [ article_type_distribution ]
[ schema ]
  🟢 sectors_as_string: 0 — Stored as string repr of list, needs ast.literal_eval


In [4]:
save_report(report, '../data/reports/quality_report.json')
print(f"Quality Score: {report['quality_score']}/100")


Report saved: ../data/reports/quality_report.json
Quality Score: 43/100


## 3. Vizualizatsiya problem

In [5]:
fig, ax = plt.subplots(figsize=(10, 4))
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    nulls.plot(kind='bar', ax=ax, color='#e07070', edgecolor='white')
    ax.set_title('Missing values by column', fontsize=13)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for i, v in enumerate(nulls.values):
        ax.text(i, v + 0.3, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
else:
    ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14)
plt.tight_layout()
plt.savefig('../data/reports/fig_missing.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved fig_missing.png')


Saved fig_missing.png


In [6]:
score = report['quality_score']
fig, ax = plt.subplots(figsize=(6, 3))
color = '#70b070' if score >= 70 else '#e0a050' if score >= 50 else '#e07070'
ax.barh(['Quality Score'], [score], color=color, edgecolor='white')
ax.barh(['Quality Score'], [100 - score], left=[score], color='#eeeeee', edgecolor='white')
ax.set_xlim(0, 100)
ax.set_title(f'Overall Quality Score: {score}/100', fontsize=14)
ax.text(score + 1, 0, str(score), va='center', fontsize=14, fontweight='bold')
ax.set_xlabel('Score')
plt.tight_layout()
plt.savefig('../data/reports/fig_quality_score.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved fig_quality_score.png')


Saved fig_quality_score.png


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

type_counts = df['article_type'].value_counts(dropna=False)
colors = ['#e07070' if str(t) == 'advertising' else '#6699cc' for t in type_counts.index]
type_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Article types (red = noise)', fontsize=12)
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=40)
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=8)

axes[1].hist(df['body'].str.len().clip(upper=2000), bins=50, color='#6699cc', edgecolor='white')
axes[1].axvline(20, color='red', linestyle='--', label='threshold = 20 chars')
axes[1].set_title('Body length distribution', fontsize=12)
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/reports/fig_quality_check.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved fig_quality_check.png')


Saved fig_quality_check.png


## 4. Detalny razbor problem

In [8]:
no_title = df[df['title'] == 'no title']
print(f'Rows without title: {len(no_title)} ({len(no_title)/len(df)*100:.1f}%)')
print(f'Unique bodies among them: {no_title["body"].nunique()}')
print(f'Sources: {no_title["source"].value_counts().to_dict()}')
print('\nSamples:')
for b in no_title['body'].head(3):
    print(f'  {repr(b[:100])}')


Rows without title: 11021 (73.3%)
Unique bodies among them: 10473
Sources: {'rdv': 5721, 't_invest': 4498, 't_analytic': 802}

Samples:
  '"После падения добычи в 2022-2023 годах Газпром добывает газа в 4 раза больше, чем нефти. Однако выр'
  '" Рейтинг акций комьюнити РДВ. #опрос   "'
  '"Нефть Brent упала до $71, Urals - до $60 за баррель. #шок_новости По мнению источника, падение цен '


In [9]:
short = df[df['body'].str.len() < 20]
print(f'Short texts (< 20 chars): {len(short)}')
print(short[['title', 'body', 'source']].head(8).to_string())


Short texts (< 20 chars): 29
         title                 body source
538   no title     "И отдельно НКЦ"    rdv
539   no title         "MOEX в SDN"    rdv
630   no title      " @aktivo_news"    rdv
992   no title            "498 ((("    rdv
1999  no title    "USDRUB > 100.  "    rdv
3189  no title   " О нас с вами:  "    rdv
3400  no title      "#Недвижимость"    rdv
4416  no title  "Выступает Шойгу  "    rdv


In [10]:
real = df[df['title'] != 'no title']
dups = real[real.duplicated(subset=['title', 'body'], keep=False)]
print(f'Exact duplicates (title+body): {len(dups)}')
if len(dups) > 0:
    print(dups[['title', 'source']].to_string())


Exact duplicates (title+body): 4
                                                           title source
13488  В Азии преобладают негативные настроения, нефть стабильна  finam
14565  В Азии преобладают негативные настроения, нефть стабильна  finam
14824      В Азии единой динамики не наблюдается, нефть дорожает  finam
14891      В Азии единой динамики не наблюдается, нефть дорожает  finam


In [11]:
labeled = df[df['positive_market_impact'].notna()]
vc = labeled['positive_market_impact'].value_counts()
print('Class balance (labeled rows):')
for label, cnt in vc.items():
    print(f'  {int(label)}: {cnt} ({cnt/len(labeled)*100:.1f}%)')


Class balance (labeled rows):
  0: 7649 (51.0%)
  1: 7351 (49.0%)


## 5. Summary table of issues

In [12]:
summary = [
    {'Issue': 'Rows without labels (scraped)', 'Count': 40, 'Pct': '0.3%', 'Severity': 'Medium', 'Fix': 'Auto-label in step 3'},
    {'Issue': 'no title entries', 'Count': 11021, 'Pct': '73.3%', 'Severity': 'Medium', 'Fix': 'Use body only for classification'},
    {'Issue': 'Advertising articles', 'Count': 1213, 'Pct': '8.1%', 'Severity': 'Medium', 'Fix': 'Remove in Strategy A, keep in Strategy B'},
    {'Issue': 'Short body (< 20 chars)', 'Count': 29, 'Pct': '0.2%', 'Severity': 'Medium', 'Fix': 'Drop from both strategies'},
    {'Issue': 'Exact duplicates (title+body)', 'Count': 2, 'Pct': '0.01%', 'Severity': 'High', 'Fix': 'Keep first occurrence'},
    {'Issue': 'Emoji in titles', 'Count': 3, 'Pct': '0.02%', 'Severity': 'Low', 'Fix': 'Clean with regex'},
    {'Issue': 'sectors/tickers as strings', 'Count': 15000, 'Pct': '100%', 'Severity': 'Low', 'Fix': 'ast.literal_eval when needed'},
]
pd.DataFrame(summary)


,Issue,Count,Pct,Severity,Fix
0,Rows without labels (scraped),40,0.3%,Medium,Auto-label in step 3
1,no title entries,11021,73.3%,Medium,Use body only for classification
2,Advertising articles,1213,8.1%,Medium,"Remove in Strategy A, keep in Strategy B"
3,Short body (< 20 chars),29,0.2%,Medium,Drop from both strategies
4,Exact duplicates (title+body),2,0.01%,High,Keep first occurrence
5,Emoji in titles,3,0.02%,Low,Clean with regex
6,sectors/tickers as strings,15000,100%,Low,ast.literal_eval when needed
